# Size ladder: how accurate, how big, how fast?

First experiment of the on-device checker. No training: we take the published span detectors,
from 17M to 400M parameters, and measure for each

1. **word-level accuracy** (span F1 on the 2,700 RAGTruth test answers, plain 0.5 cut-off),
2. **what 8-bit costs** (F1 on 300 fixed test answers, full precision against 8-bit),
3. **size** (ONNX file, full precision and 8-bit),
4. **speed** (time per answer on the CPU, one answer at a time, 1 and 2 threads).

| Step | Needs | Time on a free T4 session (estimate) |
|---|---|---|
| 1. Find the models | internet | 1 min |
| 2. Accuracy | GPU | 30-40 min |
| 3. 8-bit and speed | CPU (any runtime) | 60-90 min |
| 4. Table and chart | nothing | 1 min |

Select **Runtime > Change runtime type > T4 GPU**, then run the cells in order. Results go to
`MyDrive/ondevice_hallu/work/results`. Finished steps are skipped, so if the session stops,
run the first two cells again and then the step that was running.

**Redoing only the 8-bit part** (needed once after the 16 October 2026 fix): run the first two
cells, then Step 3 and Step 4. Step 3 sees that the old 8-bit results were made the old way
and redoes them; the accuracy results of Step 2 are kept, so no GPU is needed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, subprocess, shutil, zipfile
from pathlib import Path
from IPython.display import Markdown, Image, display

REPO = 'https://github.com/gbirhanu/ondevice-hallu.git'   # used once you have pushed the code to GitHub
CODE = Path('/content/ondevice_hallu')
DRIVE = Path('/content/drive/MyDrive/ondevice_hallu')
WORK = DRIVE / 'work'                       # results and predictions (small files)
ONNX = Path('/content/onnx')                # ONNX files stay on Colab's disk, not on Drive
DATA = Path('/content/RAGTruth')

def sh(command):
    result = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                            env={**os.environ, 'GIT_TERMINAL_PROMPT': '0'})
    print(result.stdout[-3000:], flush=True)
    return result.returncode

if (CODE / '.git').exists():
    sh(['git', '-C', str(CODE), 'pull', '--ff-only'])
elif sh(['git', 'ls-remote', REPO]) == 0:
    sh(['git', 'clone', REPO, str(CODE)])
elif (DRIVE / 'ondevice_hallu_code.zip').exists():
    print('GitHub repository not found; using the code zip on Drive.')
    with zipfile.ZipFile(DRIVE / 'ondevice_hallu_code.zip') as z:
        z.extractall('/content')
else:
    raise RuntimeError(f'Cannot reach {REPO}. Push the code to GitHub first (README, "Run it on Colab"), '
                       'and make the repository public so Colab can clone it.')
if not DATA.exists():
    sh(['git', 'clone', '--depth', '1', 'https://github.com/ParticleMedia/RAGTruth.git', str(DATA)])
sh([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==4.57.6', 'onnx', 'onnxruntime==1.29.0',
    'onnx-ir==1.0.0', 'psutil'])
WORK.mkdir(parents=True, exist_ok=True)
print('Code:', CODE, '| Results:', WORK)

In [ ]:
def run(*arguments):
    """Runs one run.py command and shows its output as it comes."""
    command = [sys.executable, '-u', str(CODE / 'run.py'), *map(str, arguments),
               '--data-dir', str(DATA), '--work-dir', str(WORK), '--onnx-dir', str(ONNX)]
    process = subprocess.Popen(command, cwd=CODE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                               env={**os.environ, 'TOKENIZERS_PARALLELISM': 'false'})
    for line in process.stdout:
        if 'Loading weights' not in line and 'TracerWarning' not in line:
            print(line, end='', flush=True)
    if process.wait():
        raise RuntimeError(f'Command failed: {" ".join(map(str, arguments))}')

def names():
    return list(json.loads((WORK / 'results' / 'models.json').read_text()))

## Step 1: find the published models

Looks up the TinyLettuce (17M, 32M, 68M) and LettuceDetect (base 150M, large 400M) models on
Hugging Face. A model that is not found is skipped.

In [ ]:
run('models')

## Step 2: word-level accuracy (GPU)

Span F1 on all 2,700 RAGTruth test answers, overall and for QA, summaries and data-to-text.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU for this step.'
for name in names():
    run('score', '--name', name)

## Step 3: 8-bit and speed (CPU)

For each model: export to ONNX, make the 8-bit version, compare their F1 on 300 fixed test
answers, and time 50 answers one at a time with 1 and 2 threads (a free Colab machine has 2
CPU cores). LettuceDetect-large is too big for a phone anyway, so it gets 100 and 20 answers
to save time; it still takes about 30 minutes.

The 8-bit file keeps the weights in 8 bits and does the arithmetic in float (recipe `w8`, see
`odh/cpu.py`). The export line prints how close the 8-bit file is to the float file, and how
many layers receive inputs with extreme values (the reason the first 8-bit attempt failed).

In [ ]:
N_CHECK, N_TIME = {'ld_large': 100}, {'ld_large': 20}     # answers; others use 300 and 50
for name in names():
    run('export', '--name', name)
    run('check8bit', '--name', name, '--n', N_CHECK.get(name, 300))
    run('bench', '--name', name, '--n', N_TIME.get(name, 50), '--threads', '1', '2', '--where', 'colab')

## Step 4: table and chart

Paste this table back into the chat.

In [ ]:
run('table')
display(Markdown((WORK / 'results' / 'table.md').read_text()))
if (WORK / 'results' / 'ladder.png').exists():
    display(Image(str(WORK / 'results' / 'ladder.png')))

## Optional: keep the small 8-bit models for phone tests

Copies the 8-bit ONNX file and tokenizer of every model of 200 MB or less to
`MyDrive/ondevice_hallu/phone_models`, for timing on a laptop or phone later. (Files copied
there before the 16 October 2026 fix are the broken 8-bit ones; this replaces them.)

In [ ]:
target = DRIVE / 'phone_models'
for name in names():
    int8 = ONNX / name / 'model_int8.onnx'
    if int8.exists() and int8.stat().st_size <= 200 * 2**20:
        (target / name).mkdir(parents=True, exist_ok=True)
        for f in (ONNX / name).iterdir():
            if f.name != 'model_fp32.onnx':
                shutil.copy(f, target / name / f.name)
        print('kept', name, round(int8.stat().st_size / 2**20, 1), 'MB')